# Business-oriented exploratory analysis

This notebook is a thin consumer of production modules. It does not define feature or target logic. Run `churn-platform stage ingest --source uci` and `churn-platform stage features --source uci` first.

In [ ]:
import pandas as pd

from churn_platform.config import load_data_config, project_path

config = load_data_config()
transactions = pd.read_parquet(project_path(config.normalized_path))
snapshots = pd.read_parquet(project_path(config.snapshots_path))

In [ ]:
quality = {
    "rows": len(transactions),
    "customers": transactions["customer_id"].nunique(),
    "start": transactions["invoice_date"].min(),
    "end": transactions["invoice_date"].max(),
    "cancellation_rate": transactions["is_cancellation"].mean(),
}
quality

In [ ]:
(
    snapshots.groupby(["split", "cutoff_date"])
    .agg(
        customers=("customer_id", "size"),
        churn_rate=("churn", "mean"),
        median_recency=("recency_days", "median"),
        median_value=("monetary_value", "median"),
    )
    .reset_index()
)